In [28]:
# ============================================================================
# Physics-Informed Fourier Neural Operator (PI-FNO) demo using the FOLAX library
# ----------------------------------------------------------------------------
# Goal:
#   Learn an OPERATOR that maps a spatial heterogeneity field K(x,y) to the
#   displacement solution U(x,y) = [Ux(x,y), Uy(x,y)] for a 2D linear elasticity
#   problem — WITHOUT supervised labels — by minimizing a weighted FE residual loss.
#
# What FOL provides in this example:
#   1) A "control" object that generates spatial fields K from low-dim parameters
#      (here: Fourier coefficients).
#   2) A physics loss object (FE-based) that computes equilibrium
#      (includign strongly applied BC).
#   3) A training wrapper that connects: coeffs -> K -> FNO -> U -> physics loss.
# ============================================================================

In [ ]:
# ----------------------------------------------------------------------------
# 0) Basic imports + create clean working directory for outputs
# ----------------------------------------------------------------------------
import sys,os
from fol.tools.usefull_functions import *

# Where all artifacts go: checkpoints, plots, exported VTK, logs, etc.
working_directory_name = 'pi_fno_mechanical_2D'
case_dir = os.path.join('.', working_directory_name)
create_clean_directory(working_directory_name)

In [ ]:
# ----------------------------------------------------------------------------
# 1) Define the mechanical PDE problem and create FE mesh + FE-based loss
# ----------------------------------------------------------------------------
from fol.loss_functions.mechanical import MechanicalLoss2DQuad

# Problem settings (domain + Dirichlet BCs)
# L: domain side length
# N: resolution used to represent K(x,y) on a grid (and usually used for mesh)
# Ux_left/right, Uy_left/right: prescribed displacements on left/right boundaries
model_settings = {"L":1,"N":30,
                "Ux_left":0.0,"Ux_right":0.05,
                "Uy_left":0.0,"Uy_right":0.05}

# Create a 2D square FE mesh. Internally, creates nodes/elements
# consistent with a (structured) quad mesh on [0,L]x[0,L].
fe_mesh = create_2D_square_mesh(L=1,N=30)

# Dirichlet boundary conditions dictionary:
#   component -> side -> value
# This tells the mechanical loss what displacement is fixed on boundaries.
bc_dict = {"Ux":{"left":model_settings["Ux_left"],"right":model_settings["Ux_right"]},
            "Uy":{"left":model_settings["Uy_left"],"right":model_settings["Uy_right"]}}

# Linear elastic material parameters
material_dict = {"young_modulus":1,"poisson_ratio":0.3}

# Create an FE-based physics loss:
# MechanicalLoss2DQuad evaluates the 2D elasticity weighted residual using quadrilateral
# elements and numerical integration (Gauss points).
mechanical_loss_2d = MechanicalLoss2DQuad("mechanical_loss_2d",loss_settings={"dirichlet_bc_dict":bc_dict,
                                                                            "num_gp":2,
                                                                            "material_dict":material_dict},
                                                                            fe_mesh=fe_mesh)

# Initialize builds internal FE data structures:
# - connectivity, shape functions, gauss points/weights
# - boundary node sets for Dirichlet BC
# - material matrices, etc.
mechanical_loss_2d.Initialize()

2025-12-29 16:15:13 - Info : mechanical_loss_2d.Initialize - for the proper batching of elements, the batch size is changed from 42 to 29


In [ ]:
# ----------------------------------------------------------------------------
# 2) Define "control": parameterize the input field K(x,y) using Fourier modes
# ----------------------------------------------------------------------------

from fol.controls.fourier_control import FourierControl

# fourier control
fourier_control_settings = {"x_freqs":np.array([2,4,6]),"y_freqs":np.array([2,4,6]),"z_freqs":np.array([0]),
                            "beta":20,"min":1e-1,"max":1}
fourier_control = FourierControl("fourier_control",fourier_control_settings,fe_mesh)
fourier_control.Initialize()
coeffs_matrix,K_matrix = create_random_fourier_samples(fourier_control,100)

2025-12-29 16:15:15 - Info : fourier_control.Initialize - finished in 0.0048 seconds


In [22]:
from fol.deep_neural_networks.ported_fourier_neural_operator_networks.fno import FNO
from flax import nnx
import jax

fno_model = FNO(
    in_channels=1,
    out_channels=2,
    hidden_channels=64,
    n_modes=(12,12),
    n_layers=4,
    rngs=nnx.Rngs(0)
)

params = nnx.state(fno_model, nnx.Param)
total_params  = sum(np.prod(x.shape) for x in jax.tree_util.tree_leaves(params))
print(f"FNO trainable parameters:{total_params}")

# initialize the FNO with some samples
init_out = fno_model(K_matrix[0:8].reshape(8,model_settings["N"],model_settings["N"],1))

FNO trainable parameters:1427266


In [23]:
import optax
from fol.deep_neural_networks.fourier_parametric_operator_learning import PhysicsInformedFourierParametricOperatorLearning

num_epochs = 10000
optimizer = optax.chain(optax.adam(1e-6))

# create fno-fol
pi_fno_pr_learning = PhysicsInformedFourierParametricOperatorLearning(name="pi_fno_pr_learning",
                                                                        control=fourier_control,
                                                                        loss_function=mechanical_loss_2d,
                                                                        flax_neural_network=fno_model,
                                                                        optax_optimizer=optimizer)

pi_fno_pr_learning.Initialize()

2025-12-29 16:15:27 - Info : pi_fno_pr_learning.Initialize - finished in 0.3102 seconds


In [24]:
train_start_id = 0
train_end_id = 80
test_start_id = 80
test_end_id = 100
#here we train for single sample at eval_id but one can easily pass the whole coeffs_matrix
pi_fno_pr_learning.Train(train_set=(coeffs_matrix[train_start_id:train_end_id,:],),
                        test_set=(coeffs_matrix[test_start_id:test_end_id,:],),
                        test_frequency=1000,
                        batch_size=8,
                        convergence_settings={"num_epochs":num_epochs,"relative_error":1e-100,"absolute_error":1e-100},
                        plot_settings={"plot_save_rate":1000},
                        train_checkpoint_settings={"least_loss_checkpointing":True,"frequency":1000},
                        working_directory=case_dir)

2025-12-29 16:15:29 - Info : pi_fno_pr_learning.Train - convergence settings:{'num_epochs': 10000, 'convergence_criterion': 'total_loss', 'relative_error': 1e-100, 'absolute_error': 1e-100}
2025-12-29 16:15:29 - Info : pi_fno_pr_learning.Train - plot settings:{'plot_list': ['total_loss'], 'plot_frequency': 1, 'save_frequency': 100, 'save_directory': './pi_fno_mechanical_2D', 'test_frequency': 1000}
2025-12-29 16:15:29 - Info : pi_fno_pr_learning.Train - restore settings:{'restore': False, 'state_directory': './pi_fno_mechanical_2D/flax_state'}
2025-12-29 16:15:29 - Info : pi_fno_pr_learning.Train - train checkpoint settings:{'least_loss_checkpointing': True, 'least_loss': inf, 'frequency': 1000, 'state_directory': './pi_fno_mechanical_2D/flax_train_state'}
2025-12-29 16:15:29 - Info : pi_fno_pr_learning.Train - test checkpoint settings:{'least_loss_checkpointing': False, 'least_loss': inf, 'frequency': 100, 'state_directory': './pi_fno_mechanical_2D/flax_test_state'}
2025-12-29 16:15:2

  0%|          | 0/10000 [00:00<?, ?it/s]/home/reza/Projects/folax/FNO/FNO-py-env/lib/python3.12/site-packages/jax/_src/lax/lax.py:5482: ComplexWarning: Casting complex values to real discards the imaginary part
  x_bar = _convert_element_type(x_bar, x.aval.dtype, x.aval.weak_type)
/home/reza/Projects/folax/FNO/FNO-py-env/lib/python3.12/site-packages/jax/_src/interpreters/mlir.py:1280: UserWarning: Some donated buffers were not usable: float32[80,10].
See an explanation at https://docs.jax.dev/en/latest/faq.html#buffer-donation.
  warnings.warn("Some donated buffers were not usable:"
 10%|█         | 1008/10000 [00:23<04:16, 35.04it/s, train_loss=0.004057283, test_loss=0.005039194] 

2025-12-29 16:15:53 - Info : pi_fno_pr_learning.train_loop - train total_loss improved from inf to 0.004154497291892767
2025-12-29 16:15:53 - Info : pi_fno_pr_learning.SaveCheckPoint - train flax nnx state is saved to ./pi_fno_mechanical_2D/flax_train_state


 20%|██        | 2011/10000 [00:39<02:54, 45.77it/s, train_loss=0.0013388043, test_loss=0.0016206343]

2025-12-29 16:16:09 - Info : pi_fno_pr_learning.train_loop - train total_loss improved from 0.004154497291892767 to 0.001344103366136551
2025-12-29 16:16:09 - Info : pi_fno_pr_learning.SaveCheckPoint - train flax nnx state is saved to ./pi_fno_mechanical_2D/flax_train_state


...

2025-12-29 16:17:46 - Info : pi_fno_pr_learning.train_loop - train total_loss improved from 0.0008822072413749993 to 0.0008665656787343323
2025-12-29 16:17:46 - Info : pi_fno_pr_learning.SaveCheckPoint - train flax nnx state is saved to ./pi_fno_mechanical_2D/flax_train_state


 90%|█████████ | 9008/10000 [02:32<00:26, 37.48it/s, train_loss=0.00085484126, test_loss=0.0010533947]

2025-12-29 16:18:02 - Info : pi_fno_pr_learning.train_loop - train total_loss improved from 0.0008665656787343323 to 0.0008548845653422177
2025-12-29 16:18:02 - Info : pi_fno_pr_learning.SaveCheckPoint - train flax nnx state is saved to ./pi_fno_mechanical_2D/flax_train_state


100%|██████████| 10000/10000 [02:48<00:00, 59.23it/s, train_loss=0.0008458822, test_loss=0.0010435261]

2025-12-29 16:18:18 - Info : pi_fno_pr_learning.train_loop - train total_loss improved from 0.0008548845653422177 to 0.0008458822267130017
2025-12-29 16:18:18 - Info : pi_fno_pr_learning.SaveCheckPoint - train flax nnx state is saved to ./pi_fno_mechanical_2D/flax_train_state
2025-12-29 16:18:18 - Info : pi_fno_pr_learning.SaveCheckPoint - final flax nnx state is saved to ./pi_fno_mechanical_2D/flax_final_state
2025-12-29 16:18:18 - Info : pi_fno_pr_learning.Train - finished in 169.0988 seconds


In [25]:
# load the final checkpoint and infer the trained model for all samples (train+test)
pi_fno_pr_learning.RestoreState(restore_state_directory=case_dir+"/flax_final_state")
FNO_UVs = pi_fno_pr_learning.Predict(coeffs_matrix)

2025-12-29 16:18:41 - Info : pi_fno_pr_learning.RestoreState - flax nnx state is restored from ./pi_fno_mechanical_2D/flax_final_state
2025-12-29 16:18:45 - Info : pi_fno_pr_learning.Predict - finished in 4.2391 seconds


In [26]:
# create and initialize FE linear solver
from fol.solvers.fe_linear_residual_based_solver import FiniteElementLinearResidualBasedSolver
fe_setting = {"linear_solver_settings":{"solver":"JAX-bicgstab","tol":1e-6,"atol":1e-6,
                                            "maxiter":1000,"pre-conditioner":"ilu"}}
fe_solver = FiniteElementLinearResidualBasedSolver("lin_fe_solver",mechanical_loss_2d,fe_setting)
fe_solver.Initialize()

2025-12-29 16:18:47 - Info : lin_fe_solver.__init__ - finished in 0.0000 seconds
2025-12-29 16:18:47 - Info : lin_fe_solver.Initialize - finished in 0.0000 seconds


In [27]:
# now compare 
def plot_set_results(set_ids:np.array,set_name:str):

    for eval_id in set_ids:
        FNO_UV = np.asarray(FNO_UVs[eval_id]).flatten()
        fe_mesh[f'FNO_UV_{eval_id}'] = FNO_UV.reshape((fe_mesh.GetNumberOfNodes(), 2))

        # solve FE here
        FE_UV = np.asarray(fe_solver.Solve(K_matrix[eval_id],np.zeros(2*fe_mesh.GetNumberOfNodes()))).flatten()

        fe_mesh[f'FE_UV_{eval_id}'] = FE_UV.reshape((fe_mesh.GetNumberOfNodes(), 2))

        absolute_error = abs(FNO_UV- FE_UV)
        fe_mesh[f'abs_error_{eval_id}'] = absolute_error.reshape((fe_mesh.GetNumberOfNodes(), 2))
        
        plot_mesh_vec_data(model_settings["L"], [K_matrix[eval_id,:],FNO_UV[::2],FE_UV[::2],absolute_error[::2]], 
                        subplot_titles= ['Heterogeneity', 'FNO_U', 'FE_U', "absolute_error"], fig_title=None, cmap='viridis',
                            block_bool=True, colour_bar=True, colour_bar_name=None,
                            X_axis_name=None, Y_axis_name=None, show=False, file_name=os.path.join(case_dir,f'{set_name}_{eval_id}_U.png'))
        
        
        plot_mesh_vec_data(model_settings["L"], [K_matrix[eval_id,:],FNO_UV[1::2],FE_UV[1::2],absolute_error[1::2]], 
                        subplot_titles= ['Heterogeneity', 'FNO_V', 'FE_V', "absolute_error"], fig_title=None, cmap='viridis',
                            block_bool=True, colour_bar=True, colour_bar_name=None,
                            X_axis_name=None, Y_axis_name=None, show=False, file_name=os.path.join(case_dir,f'{set_name}_{eval_id}_V.png'))

plot_set_results(np.arange(train_start_id,train_end_id,30),"train")
plot_set_results(np.arange(test_start_id,test_end_id,10),"test")

# now export results as vtk (square.vtk)
fe_mesh.Finalize(export_dir=case_dir)

2025-12-29 16:18:50 - Info : mechanical_loss_2d.ApplyDirichletBCOnDofVector - finished in 0.0368 seconds
2025-12-29 16:18:51 - Info : mechanical_loss_2d.ComputeJacobianMatrixAndResidualVector - finished in 0.9392 seconds
2025-12-29 16:18:51 - Info : lin_fe_solver.JaxBicgstabLinearSolver - finished in 0.5518 seconds
2025-12-29 16:18:51 - Info : lin_fe_solver.Solve - finished in 1.5623 seconds
2025-12-29 16:18:52 - Info : mechanical_loss_2d.ApplyDirichletBCOnDofVector - finished in 0.0001 seconds
2025-12-29 16:18:52 - Info : mechanical_loss_2d.ComputeJacobianMatrixAndResidualVector - finished in 0.0022 seconds
2025-12-29 16:18:52 - Info : lin_fe_solver.JaxBicgstabLinearSolver - finished in 0.2445 seconds
2025-12-29 16:18:52 - Info : lin_fe_solver.Solve - finished in 0.2473 seconds
2025-12-29 16:18:54 - Info : mechanical_loss_2d.ApplyDirichletBCOnDofVector - finished in 0.0001 seconds
2025-12-29 16:18:54 - Info : mechanical_loss_2d.ComputeJacobianMatrixAndResidualVector - finished in 0.00

Warning: VTK requires 3D vectors, but 2D vectors given. Appending 0 third component to FNO_UV_0.

Warning: VTK requires 3D vectors, but 2D vectors given. Appending 0 third component to FE_UV_0.

Warning: VTK requires 3D vectors, but 2D vectors given. Appending 0 third component to abs_error_0.

Warning: VTK requires 3D vectors, but 2D vectors given. Appending 0 third component to FNO_UV_30.

...

Warning: VTK requires 3D vectors, but 2D vectors given. Appending 0 third component to abs_error_80.

Warning: VTK requires 3D vectors, but 2D vectors given. Appending 0 third component to FNO_UV_90.

Warning: VTK requires 3D vectors, but 2D vectors given. Appending 0 third component to FE_UV_90.

Warning: VTK requires 3D vectors, but 2D vectors given. Appending 0 third component to abs_error_90.

2025-12-29 16:18:58 - Info : square_io.Finalize - finished in 0.0250 seconds
